# ShallowLandslider: Quick‑start Notebook

This notebook is a **template** to help you understand and use the `ShallowLandslider` modelling component on a Landlab grid.
- ⚠️ Check placeholder paths and parameters before running end‑to‑end."


## Prerequisites

- Python 3.9+ recommended
- Packages: `numpy`$\geq$ 2.0, `matplotlib`, `pandas`, `landlab`, `scipy`, `scikit-image`, `richdem`, `seaborn`, 
- Local modules available on your path: `shallow_landslide_component.py`, and `helper_functions.py` providing:
  - `get_topo`, `apply_soil_depth`, `calculate_terrain_attribute`, `generate_acceleration_grid`,
  - `pickle_or_not_to_pickle` (if using measured data)

> If running in a clean environment, install deps, e.g.:
```bash
pip install landlab numpy pandas matplotlib scipy scikit-image seaborn richdem
```

> Ensure the working directory contains your component and helper modules, or update `sys.path` accordingly.

> The component has been tested on RasterModelGrid **only**; it might not work with other types of grids

## Imports & Settings

In [ ]:
# Standard imports
import os
import numpy as np
import matplotlib.pyplot as plt

# Landlab
from landlab.components import PriorityFloodFlowRouter

# Project modules (ensure these files are in the working directory or in PYTHONPATH)
from shallow_landslide_component import ShallowLandslider

# Utilities to help quick-start
from utils import (
    get_topo,
    apply_soil_depth,
    calculate_terrain_attribute,
    generate_acceleration_grid,
    pickle_or_not_to_pickle,
    plot_comparison_panels_with_ecdf
)

# Plotting aesthetics
plt.style.use('seaborn-v0_8')


## Configuration (edit these for your study area)

In [ ]:
# DEM & flow settings
config = {
    'dem_info': {
        'dem_type': 'SRTMGL1',
        'north': 27.94, 'east': 85.98, 'south': 27.82, 'west': 85.82,
        'buffer': 0.01,
        'smooth_num': 4,
        'plot_dem': False,
        'api_key': 'f08b2664772eb044626d5cb114924de1' # required to download DEMs from OpenTopography
    },
    'flow_params': {
        'flow_metric': 'D8',
        'separate_hill_flow': True,
        'depression_handling': 'fill',
        'update_hill_depressions': True,
        'accumulate_flow': True,
    },
    'soil_params': {
        'angle_int_frict': 30,      # degrees
        'cohesion_eff': 15e3,       # Pa
        'submerged_soil_proportion': 0.5,
        'max_soil_depth': 1.5,      # m
        'plot_soil': False,
        'distribution': 'curvature', # 'uniform'|'elevation'|'curvature'|'drainage_area'|'mean_elev_curv'
        'relationship': 'linear_std_local',
        # Elevation-based params
        'decay_rate': 1.0,
        'exponent': 2.0,
        # Drainage-area based params
        'drainage_transform': 'threshold',
        'drainage_threshold': 1e6,
        'drainage_power': 0.3,
        # Curvature-based params
        'P0': 0.05, 'h_star': 1.0, 'D': 0.01, 'h_min': 0.1, 'h_no_ss': 0.0,
    },
    'pga': {
        'horizontal_max': 0.6,
        'vertical_max': 0.2,
        'distribution': 'uniform',
        'plot_grids': False,
    },
    'simulation': {
        'time_shaking': 10, # seconds (used for Newmark displacement if enabled)
        'displacement_threshold': 0.0,
        'aspect_interval': 20, # interval for topographic aspect zones
        'random_seed': 5000,
        'handle_small_regions': 'merge',
        'selection_method': 'probabilistic', # or 'pga_weighted'
        'proportion_method': 'conservative', # 'empirical'|'statistical'|'risk_profile'|'adaptive'|'conservative'
        'compute_displacement': False,
    },
    'output': {
        'verbose': False,
        'save_plots': False,
        'output_dir': './outputs',
        'save_pickle': False,
        'load_pickle': False,
    }
}

# Ensure output dir exists
os.makedirs(config['output']['output_dir'], exist_ok=True)


## Load measured landslide data for KDE‑guided splitting
- pickle_or_not_to_pickle bundles all of the input measured data for quick access, and allows the model to pick up the required KDE data for splitting later.
    - Once a pickle has been made with a certain name (e.g., 'measured_data.pkl'), the function will pick it up and open it directly. This process takes milliseconds.
    - If you want to change this, just change the name of the pickle in the pickle_path argument below.
- The pointers need to be pointed towards the files containing the data. Currently they point towards example data from Nepal to run this notebook. 

In [ ]:
# If you have measured landslide data/KDEs, configure the paths here.

file_name_dict = {
    'file1': '/path/to/measuredLandslides_all.csv', # All mapped landslides in region, with area, length_m, and width_m fields
    'file2': '/path/to/measuredLandslides_all_spatialStats.csv', # Zonal stats for all measured landslides
    'file3': '/path/to/region_specific_spatialStats.csv', # Zonal stats for any subregion being tested
}

kde_dict = None
bundle = pickle_or_not_to_pickle(file_name_dict=file_name_dict,
                                    pickle_path=os.path.join('./utils', 'measured_data.pkl'))
kde_dict = {
    'kde_data': bundle.get('kde_data'),
    'kde_transform': bundle.get('kde_transform'),
}

print('Loaded measured KDEs.')

In [ ]:
measured_spatial_stats_clipped = bundle.get("measured_spatial_stats_clipped")

# Dataframe containing measured zonal statistics for subregion being tested
measured_spatial_stats_clipped_filt = measured_spatial_stats_clipped[measured_spatial_stats_clipped["Area_m2"]>900]

## Build Landlab grid, route flow, and derive terrain attributes
- Uses get_topo() and bmi-topography to download and set up a DEM from OpenTopo for you to help quickly set up a grid. If you have your own grid, set it up here.
- apply_soil_depth() helps set up a simulated soil depth distribution
    - Using a curvature-based distribution here requires richdem and calculate_terrain_attribute()
- If these already exist, you can skip the next several cells

In [ ]:
# Get DEM and build grid
mg, z = get_topo(
    dem_type=config['dem_info']['dem_type'],
    north=config['dem_info']['north'],
    south=config['dem_info']['south'],
    east=config['dem_info']['east'],
    west=config['dem_info']['west'],
    buffer=config['dem_info']['buffer'],
    api_key=config['dem_info']['api_key'],
)

### Flow routing

In [ ]:
pf = PriorityFloodFlowRouter(
    mg,
    flow_metric=config['flow_params']['flow_metric'],
    separate_hill_flow=config['flow_params']['separate_hill_flow'],
    depression_handler=config['flow_params']['depression_handling'],
    update_hill_depressions=config['flow_params']['update_hill_depressions'],
    accumulate_flow=config['flow_params']['accumulate_flow'],
)
pf.run_one_step()

### Calculate planform curvature

In [ ]:
# Terrain attribute example: planform curvature (for curvature‑based soil depth using richdem)
curv = calculate_terrain_attribute(
    grid=mg, field_name='topographic__elevation', attrib='planform_curvature'
)

### Add soil depth
- Also adds bedrock__elevation field to grid (also optional)

In [ ]:
if 'soil__depth' not in mg.at_node:
    soil_depth = mg.add_zeros('soil__depth', at='node')

soil_depth = apply_soil_depth(
    mg,
    max_soil_depth=config['soil_params']['max_soil_depth'],
    distribution=config['soil_params']['distribution'],
    relationship=config['soil_params']['relationship'],
    decay_rate=config['soil_params']['decay_rate'],
    exponent=config['soil_params']['exponent'],
    drainage_transform=config['soil_params']['drainage_transform'],
    drainage_threshold=config['soil_params']['drainage_threshold'],
    drainage_power=config['soil_params']['drainage_power'],
    P0=config['soil_params']['P0'],
    h_star=config['soil_params']['h_star'],
    D=config['soil_params']['D'],
    h_min=config['soil_params']['h_min'],
    h_no_ss=config['soil_params']['h_no_ss'],
    plot=config['soil_params']['plot_soil'],
)

# Bedrock elevation from DEM minus soil thickness
if 'bedrock__elevation' not in mg.at_node:
    mg.add_zeros('bedrock__elevation', at='node', clobber=True)
mg.at_node['bedrock__elevation'][:] = mg.at_node['topographic__elevation'] - mg.at_node['soil__depth']


### Calculates slopes and topographic aspect

In [ ]:
slopes_rad = mg.calc_slope_at_node(elevs='topographic__elevation')
slopes_deg = np.degrees(slopes_rad)
aspect_nodes = np.array(mg.calc_aspect_at_node(elevs='topographic__elevation',
                                                unit='degrees', ignore_closed_nodes=True))
aspect_nodes[mg.boundary_nodes] = np.nan

## Generate earthquake PGA grids
- Quickly sets up arrays for PGA_h and PGA_v using parameters set up above
- These can be "uniform", "circular", "square", "diamond", or "exponential" to test various patterns
- Can also be skipped if you have existing arrays of the same size as the grid

In [ ]:
pga_h, pga_v = generate_acceleration_grid(
    grid=mg,
    horizontal_max=config['pga']['horizontal_max'],
    vertical_max=config['pga']['vertical_max'],
    distribution=config['pga']['distribution'],
    plot_grids=config['pga']['plot_grids'],
)

## Initialise and run the ShallowLandslider component
- Parameters taken from dictionary at the top of the notebook

In [ ]:
# Initialise component
ls = ShallowLandslider(
    mg,
    cohesion_eff=config['soil_params']['cohesion_eff'],
    angle_int_frict=config['soil_params']['angle_int_frict'],
    submerged_soil_proportion=config['soil_params']['submerged_soil_proportion'],
    pga_h=pga_h,
    pga_v=pga_v,
    pga_h_max=config['pga']['horizontal_max'],
    pga_v_max=config['pga']['vertical_max'],
    selection_method=config['simulation']['selection_method'],
    proportion_method=config['simulation']['proportion_method'],
    random_seed=config['simulation']['random_seed'],
    aspect_interval=config['simulation']['aspect_interval'],
    compute_displacement=config['simulation']['compute_displacement'],
    time_shaking=config['simulation']['time_shaking'],
    displacement_threshold=config['simulation']['displacement_threshold'],
    g=9.81,
    split_by_width_config=(None if kde_dict is None else {
        'kde_data': kde_dict.get('kde_data'),
        'kde_transform': kde_dict.get('kde_transform'),
        'convergence_threshold': 0.75,
        'min_region_size': 10,
        'max_iterations': 10,
        'width_threshold': 1.5,
    }),
    verbose=config['output']['verbose'],
)

In [ ]:
# Run component
ls.run_one_step()

In [ ]:
# Extract results and set up for plotting
props = ls.results.get('group_properties')
print('Group properties (head):')
print(props.head() if props is not None else 'No properties table computed.')

# Get labels and reshape to 2-D for draping
labels = mg.at_node['landslide__selected_labels'].copy().reshape(mg.shape)

# Mask out zeros
labels_masked = np.ma.masked_where(labels == 0, labels)

# Make a copy of the cmap and set masked values to transparent
cmap = plt.cm.get_cmap('jet').copy()
cmap.set_bad(alpha=0)  # masked pixels won't be visible

props_filtered = props.loc[props["selected"]]

## Visualise simulated landslides
- Plots a figure with four panels:
    1. Simulated landslides draped over hillshade
    2. Histogram and ecdf of landslide area (mapped vs. simulated)
    3. Histogram and ecdf of mean landslide elevations (mapped vs. simulated)
    4. Histogram and ecdf of mean landslide slopes (mapped vs. simulated)

In [ ]:
plot_comparison_panels_with_ecdf(
    observed_df=measured_spatial_stats_clipped_filt,
    model_df=props_filtered,
    mg=mg,                     # optional
    labels_masked=labels_masked,  # optional
    title=f"Modeled output: {config['soil_params']['distribution']} - {config['soil_params']['relationship']}",
    save_path=None             # or "comparison_panels.pdf"
)

## Troubleshooting & Tips

- **Field names** follow the Landlab convention used in the component (e.g., `topographic__elevation`, `soil__depth`).
- If you see a `ValueError` about sizes, ensure arrays match `mg.number_of_nodes`.
- If DEM fetching requires authentication/API keys, set `dem_info.api_key` appropriately.
- Use `selection_method='pga_weighted'` for a deterministic proportion based on PGA; otherwise keep `'probabilistic'`.
- Turn on `compute_displacement=True` to write `landslide__newmark_displacement`.
- Set `aspect_interval` to a coarser (e.g., 45°) if too many subgroups are created.
- For reproducibility, set `random_seed`.
- For very small regions, consider post‑processing with `handle_small_regions` in your helper functions or merge/keep logic.

> For performance on large domains, start with smaller extents and lower `smooth_num`, and avoid plotting huge arrays.
